In [1]:
# import dependencies
import pandas as pd
import numpy as np
import geopandas as gpd
from functools import reduce

Note: Probably gonna rework how the National Risk Index is saved.

Technical debt for later.

In [2]:
# open csv file
df = pd.read_csv("../data/cleaned/town_level_merged_for_eda.csv")

# open geojson file
gdf = gpd.read_file("../data/cleaned/census.geojson")

In [3]:
# select GEOID and geometry columns, and rename NAMELSAD to town_name
gdf = gdf[["GEOID", "geometry", "NAMELSAD"]]
gdf = gdf.rename(columns={"NAMELSAD": "town_name"})

# export to geojson
gdf.to_file("../docs/static/resources/town_boundaries.geojson", driver="GeoJSON")

In [4]:
# data cleaning and preprocessing

# convert town_area_sqm to area_sq_km
df["area_sq_km"] = df["town_area_sqm"] / 1e6

# cap extreme outliers in poverty variable (winsorize)
df["pct_below_poverty"] = df["pct_below_poverty"].clip(
    upper=df["pct_below_poverty"].quantile(0.99)
)

# fill funding and claims NaNs with 0
cols_to_fill = [
    "federalShareObligated_adj",
    "funding_per_capita",
    "log_funding_per_capita",
    "funding_per_occupied_unit",
    "log_funding_per_occupied_unit",
    "claims_paid_per_capita",
    "current_insurance_penetration",
]
df[cols_to_fill] = df[cols_to_fill].fillna(0)

# mark towns with no population
df["valid_population"] = df["total_population"] > 0
df_zero_pop = df[~df["valid_population"]].copy()
df_valid = df[df["valid_population"]].copy()

In [5]:
# functions for building indices


def rank_normalize(df, cols):
    """
    Apply rank-based normalization (percentile ranks) to specified columns in a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        cols (list): List of column names to normalize.

    Returns:
        pd.DataFrame: DataFrame with specified columns rank-normalized.
    """
    df = df.copy()
    for col in cols:
        df[col] = df[col].rank(pct=True)
    return df


def build_index(df, risk_vars, vuln_vars, funding_var):
    """
    Build risk, vulnerability, and need indices, and compute funding gap.

    Args:
        df (pd.DataFrame): Input DataFrame.
        risk_vars (list): List of risk variable column names.
        vuln_vars (list): List of vulnerability variable column names.
        funding_var (str): Column name for funding variable.

    Returns:
        pd.DataFrame: DataFrame with new index and gap columns.
    """
    df = df.copy()

    all_vars = risk_vars + vuln_vars

    # normalize risk and vulnerability variables
    if all_vars:
        df_norm = rank_normalize(df, all_vars)
    else:
        df_norm = df.copy()

    # calculate risk and vulnerability indices as the mean of their respective (z-scored) variables
    df["risk_index"] = df_norm[risk_vars].mean(axis=1) if risk_vars else 0
    df["vuln_index"] = df_norm[vuln_vars].mean(axis=1) if vuln_vars else 0

    # combined need index (normalized 0–1-ish if rank, centered if z)
    df["need_index"] = (df["risk_index"] + df["vuln_index"]) / 2

    # # scale funding variable to 0–1 range using rank normalization, so it's on the same scale as the need index
    # # safeguard - I've already log-transformed funding and filled NaNs in preprocessing
    funding = df[funding_var].fillna(0).copy()
    df["funding_scaled"] = funding.rank(pct=True)

    # gap: positive => underfunded relative to need; negative => overfunded
    df["gap_index"] = df["need_index"] - df["funding_scaled"]

    return df

In [6]:
# function to assign quadrants based on need and funding indices


def add_quadrants(df):
    """
    Add quadrant labels based on need and funding indices.
        Quadrants:
        - zero_funding: no funding regardless of need
        - underfunded: high need, low funding
        - aligned: high need, high funding
        - overfunded: low need, high funding
        - low_priority: low need, low funding

    Args:
        df (pd.DataFrame): Input DataFrame with 'need_index' and 'funding_scaled' columns.

    Returns:
        pd.DataFrame: DataFrame with an additional 'quadrant' column.
    """
    df = df.copy()

    # calculate median for need to define quadrants
    need_med = df["need_index"].median()

    df["quadrant"] = "unassigned"

    # zero funding = separate category regardless of need
    df.loc[df["funding_scaled"] == df["funding_scaled"].min(), "quadrant"] = (
        "zero_funding"
    )

    # get mask for non-zero funding to apply quadrant logic only to those rows
    mask = df["funding_scaled"] > df["funding_scaled"].min()

    # calculate median of non-zero funding for quadrant split
    fund_med_nonzero = df.loc[mask, "funding_scaled"].median()

    # assign quadrants based on need and funding relative to their medians
    df.loc[
        mask
        & (df["need_index"] >= need_med)
        & (df["funding_scaled"] < fund_med_nonzero),
        "quadrant",
    ] = "underfunded"
    df.loc[
        mask
        & (df["need_index"] >= need_med)
        & (df["funding_scaled"] >= fund_med_nonzero),
        "quadrant",
    ] = "aligned"
    df.loc[
        mask
        & (df["need_index"] < need_med)
        & (df["funding_scaled"] >= fund_med_nonzero),
        "quadrant",
    ] = "overfunded"
    df.loc[
        mask
        & (df["need_index"] < need_med)
        & (df["funding_scaled"] < fund_med_nonzero),
        "quadrant",
    ] = "low_priority"

    return df

In [7]:
def analyze_models(df, model_specs):
    """
    Run the full analysis for each model specification, including index building, quadrant assignment, and result summarization.

    Args:
        df (pd.DataFrame): Input DataFrame with town-level data.
        model_specs (dict): Dictionary of model specifications with risk and vulnerability variable lists.

    Returns:
        dict: Dictionary containing results for each model specification.
    """
    # initialize results dictionary
    results = {}

    # loop through model specifications and run analysis
    for name, spec in model_specs.items():
        # extract variable lists from spec
        risk_vars = spec.get("risk", [])
        vuln_vars = spec.get("vuln", [])
        funding_var = "log_funding_per_capita"

        # drop any town with no population to avoid skewing indices
        df_valid = df[df["valid_population"]].copy()

        # build indices and assign quadrants
        df_results = build_index(df_valid, risk_vars, vuln_vars, funding_var)
        df_quadrants = add_quadrants(df_results)

        # store results in dictionary
        results[name] = {
            "df_results": df_results,
            "quadrants": df_quadrants,
        }

    # return the results dictionary
    return results

In [8]:
# select model specifications of interest
model_specs = {
    # single risk variable with core vulnerability variables
    "core_EAL_model": {
        "risk": ["IFLD_EALT_weighted"],
        "vuln": ["pct_below_poverty", "percent_elderly", "pct_no_vehicle"],
    },
    # EAL per capita
    "eal_per_capita_model": {
        "risk": ["EAL_per_capita"],
        "vuln": ["pct_below_poverty", "percent_elderly", "pct_no_vehicle"],
    },
    # FEMA's own risk score, for benchmarking
    "fema_national_risk_index": {
        "risk": ["RISK_SCORE_avg"],
    },
}

In [9]:
# run models
results = analyze_models(df_valid, model_specs)

In [10]:
# collect base columns for final DataFrame and rename for clarity
base_cols = [
    "GEOID",
    "town_name",
    "total_population",
    "area_sq_km",
    "pct_river_corridor",
    "valid_population",
    "has_funding",
    "federalShareObligated_adj",
    "funding_per_capita",
    "pct_below_poverty",
    "percent_elderly",
    "pct_no_vehicle",
    "claims_paid_per_capita",
    "pct_renter_occupied",
    "median_income",
    "IFLD_EALT_weighted",
    "EAL_per_capita",
]
base_df = df_valid[base_cols].copy()
base_df = base_df.rename(
    columns={
        "total_population": "population",
        "federalShareObligated_adj": "funding_total",
    }
)

# rename model-specific columns from results
model_map = {
    "core_EAL_model": "eal",
    "eal_per_capita_model": "eal_per_capita",
    "fema_national_risk_index": "nri",
}

# iterate through models, rename columns, compute ranks, and append to list for merging
model_dfs = []
for model_key, suffix in model_map.items():
    res = results[model_key]["df_results"].copy()
    quad = results[model_key]["quadrants"][["GEOID", "quadrant"]].copy()
    res = res.merge(quad, on="GEOID", how="left").rename(
        columns={
            "risk_index": f"risk_{suffix}",
            "need_index": f"need_{suffix}",
            "gap_index": f"gap_{suffix}",
            "quadrant": f"quadrant_{suffix}",
        }
    )
    # compute ranks (descending: rank 1.0 = highest/worst, consistent across all metrics)
    res[f"risk_rank_{suffix}"] = res[f"risk_{suffix}"].rank(ascending=True, pct=True)
    res[f"need_rank_{suffix}"] = res[f"need_{suffix}"].rank(ascending=True, pct=True)
    res[f"gap_rank_{suffix}"] = res[f"gap_{suffix}"].rank(ascending=True, pct=True)
    # select relevant columns
    model_dfs.append(
        res[
            [
                "GEOID",
                f"risk_{suffix}",
                f"need_{suffix}",
                f"gap_{suffix}",
                f"risk_rank_{suffix}",
                f"need_rank_{suffix}",
                f"gap_rank_{suffix}",
                f"quadrant_{suffix}",
            ]
        ]
    )

# merge all model DataFrames on GEOID
# reduce applies the merge function cumulatively to the list of DataFrames, merging them one by one on GEOID
df_merged = reduce(lambda left, right: pd.merge(left, right, on="GEOID"), model_dfs)
df_merged = pd.merge(base_df, df_merged, on="GEOID", how="left")

# compute composite vulnerability index (always the same across all models)
df_merged["vulnerability"] = (
    df_merged[["pct_below_poverty", "percent_elderly", "pct_no_vehicle"]]
    .rank(pct=True)
    .mean(axis=1)
)
df_merged["vulnerability_rank"] = df_merged["vulnerability"].rank(
    ascending=True, pct=True
)

# compute funding ranks for the final DataFrame (also not model-specific)
df_merged["funding_rank"] = (
    df_merged["funding_per_capita"]
    .where(df_merged["funding_per_capita"] > 0)
    .rank(pct=True)
    .fillna(0)
)


# relative scaling function
def to_relative_centered(series):
    mean = series.mean()
    return (series / mean) - 1  # centered at 0


# compute relative to state average columns
rel_cols = [
    "risk_eal",
    "risk_eal_per_capita",
    "risk_nri",
    "need_eal",
    "need_eal_per_capita",
    "need_nri",
    "vulnerability",
]

for col in rel_cols:
    df_merged[f"{col}_rel"] = to_relative_centered(df_merged[col])

# compute funding (log before computing relative to reduce skew)
df_merged["funding_log"] = np.log1p(df_merged["funding_per_capita"])
df_merged["funding_rel"] = to_relative_centered(df_merged["funding_log"])

# compute claims rank and relative (same log treatment as funding to reduce skew)
df_merged["claims_rank"] = (
    df_merged["claims_paid_per_capita"]
    .where(df_merged["claims_paid_per_capita"] > 0)
    .rank(pct=True)
    .fillna(0)
)
df_merged["claims_log"] = np.log1p(df_merged["claims_paid_per_capita"])
df_merged["claims_rel"] = to_relative_centered(df_merged["claims_log"])

# can't use to_relative_centered here — gap is zero-centered by construction (positive = underfunded,
# negative = overfunded), so mean(gap) ≈ 0 and dividing by it produces explosive values.
# instead, scale by the mean absolute gap: a value of 1.0 means "gap equal to the typical gap magnitude."
# this preserves the sign (zero = exactly funded to need) and spreads the distribution
# evenly without a single outlier compressing the color scale (/abs().max() would do that).
for col in ["gap_eal", "gap_eal_per_capita", "gap_nri"]:
    # df_merged[f"{col}_rel"] = df_merged[col] / df_merged[col].abs().max()
    df_merged[f"{col}_rel"] = df_merged[col] / df_merged[col].abs().mean()
    # df_merged[f"{col}_rel"] = df_merged[col] / df_merged[col].std()

# format population for better readability in the web app - do I need to do this?
df_merged["population"] = df_merged["population"].astype(int)

# reorder columns
final_cols = [
    "GEOID",
    "town_name",
    "population",
    "valid_population",
    "area_sq_km",
    "pct_river_corridor",
    "pct_below_poverty",
    "percent_elderly",
    "pct_no_vehicle",
    "pct_renter_occupied",
    "median_income",
    "IFLD_EALT_weighted",
    "EAL_per_capita",
    # raw
    "risk_eal",
    "risk_eal_per_capita",
    "risk_nri",
    "need_eal",
    "need_eal_per_capita",
    "need_nri",
    "gap_eal",
    "gap_eal_per_capita",
    "gap_nri",
    "funding_total",
    "funding_per_capita",
    "claims_paid_per_capita",
    "vulnerability",
    # relative
    "risk_eal_rel",
    "risk_eal_per_capita_rel",
    "risk_nri_rel",
    "need_eal_rel",
    "need_eal_per_capita_rel",
    "need_nri_rel",
    "gap_eal_rel",
    "gap_eal_per_capita_rel",
    "gap_nri_rel",
    "funding_rel",
    "claims_rel",
    "vulnerability_rel",
    # ranks
    "funding_rank",
    "claims_rank",
    "vulnerability_rank",
    "risk_rank_eal",
    "risk_rank_eal_per_capita",
    "risk_rank_nri",
    "need_rank_eal",
    "need_rank_eal_per_capita",
    "need_rank_nri",
    "gap_rank_eal",
    "gap_rank_eal_per_capita",
    "gap_rank_nri",
    # categorical
    "quadrant_eal",
    "quadrant_eal_per_capita",
    "quadrant_nri",
]

# ensure all final columns are present in the merged DataFrame before selecting
final_cols = [col for col in final_cols if col in df_merged.columns]
df_final = df_merged[final_cols]


# append zero population towns back to the final DataFrame with nulls for indices and gaps
df_zero_pop = df_zero_pop[base_cols].copy()
derived_cols = [col for col in df_final.columns if col not in base_cols]
df_zero_pop[derived_cols] = None
df_zero_pop["population"] = df_zero_pop["population"].fillna(0).astype(int)
df_final = pd.concat([df_final, df_zero_pop[final_cols]], ignore_index=True)

C:\Users\johbr\AppData\Local\Temp\ipykernel_20220\858735080.py:210: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_zero_pop["population"] = df_zero_pop["population"].fillna(0).astype(int)
C:\Users\johbr\AppData\Local\Temp\ipykernel_20220\858735080.py:211: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final, df_zero_pop[final_cols]], ignore_index=True)


In [11]:
# state-level summary row
state = {}

# population-weighted metrics
pop = df_final["population"].replace(0, np.nan)


def wavg(col):
    return np.nansum(df_final[col] * pop) / np.nansum(pop)


# core stats
state["town_name"] = "State of Vermont"
state["population"] = int(df_final["population"].sum())
state["IFLD_EALT_weighted"] = df_merged["IFLD_EALT_weighted"].sum()
state["EAL_per_capita"] = wavg("EAL_per_capita")
state["funding_total"] = df_final["funding_total"].sum()
state["funding_per_capita"] = state["funding_total"] / state["population"]

# weighted averages
for col in [
    "risk_eal",
    "risk_eal_per_capita",
    "risk_nri",
    "need_eal",
    "need_eal_per_capita",
    "need_nri",
    "vulnerability",
    "pct_below_poverty",
    "percent_elderly",
    "pct_no_vehicle",
    "pct_river_corridor",
    "median_income",
    "pct_renter_occupied",
]:
    state[col] = wavg(col)

# gaps (can just avg since already normalized-ish)
for col in ["gap_eal", "gap_eal_per_capita", "gap_nri"]:
    state[col] = df_final[col].mean()

# ranks: all descending (rank 1.0 = highest/worst)
# rank = fraction of towns with value ABOVE the unweighted mean
for col in [
    "risk_eal",
    "risk_eal_per_capita",
    "risk_nri",
    "need_eal",
    "need_eal_per_capita",
    "need_nri",
]:
    ref_val = df_merged[col].mean()  # same mean used in to_relative_centered
    n = df_merged[col].notna().sum()
    percentile = (df_merged[col] > ref_val).sum() / n
    base, suffix = col.split("_", 1)
    state[f"{base}_rank_{suffix}"] = percentile

# vulnerability: same pattern, no model suffix
ref_val = df_merged["vulnerability"].mean()
n = df_merged["vulnerability"].notna().sum()
state["vulnerability_rank"] = (df_merged["vulnerability"] > ref_val).sum() / n

# gap: descending (rank 1.0 = most underfunded = gap > 0 after flip)
# rank = fraction of towns with gap ABOVE 0 (more underfunded than the average town)
for col in ["gap_eal", "gap_eal_per_capita", "gap_nri"]:
    ref_val = state[col]  # mean gap already computed above
    n = df_merged[col].notna().sum()
    percentile = (df_merged[col] <= ref_val).sum() / n
    base, suffix = col.split("_", 1)
    state[f"{base}_rank_{suffix}"] = percentile

# funding: descending (rank 1.0 = most funded)
# rank = fraction of towns with funding BELOW the log-mean back-transformed value
funding_pc_log = np.log1p(
    df_merged["funding_per_capita"].where(df_merged["funding_per_capita"] > 0)
)
ref_funding_pc = np.expm1(funding_pc_log.mean())
valid = df_merged["funding_total"].notna()
state["funding_rank"] = (
    df_merged.loc[valid, "funding_per_capita"] <= ref_funding_pc
).sum() / valid.sum()

# claims: population-weighted average and rank (same log-mean pattern as funding)
state["claims_paid_per_capita"] = wavg("claims_paid_per_capita")
claims_log = np.log1p(
    df_merged["claims_paid_per_capita"].where(df_merged["claims_paid_per_capita"] > 0)
)
ref_claims_pc = np.expm1(claims_log.mean())
valid_claims = df_merged["claims_paid_per_capita"].notna()
state["claims_rank"] = (
    df_merged.loc[valid_claims, "claims_paid_per_capita"] <= ref_claims_pc
).sum() / valid_claims.sum()

# convert to DataFrame and append
df_state = pd.DataFrame([state])
df_final = pd.concat([df_final, df_state], ignore_index=True)

In [12]:
null_counts = df_final.isnull().sum()
print("Null counts in final DataFrame:")
print(null_counts)

Null counts in final DataFrame:
GEOID                       1
town_name                   0
population                  0
valid_population            1
area_sq_km                  1
pct_river_corridor          0
pct_below_poverty           6
percent_elderly             6
pct_no_vehicle              6
pct_renter_occupied         6
median_income               9
IFLD_EALT_weighted          1
EAL_per_capita              1
risk_eal                    6
risk_eal_per_capita         6
risk_nri                    6
need_eal                    6
need_eal_per_capita         6
need_nri                    6
gap_eal                     6
gap_eal_per_capita          6
gap_nri                     6
funding_total               6
funding_per_capita          0
claims_paid_per_capita      0
vulnerability               6
risk_eal_rel                7
risk_eal_per_capita_rel     7
risk_nri_rel                7
need_eal_rel                7
need_eal_per_capita_rel     7
need_nri_rel                7
gap_eal_

In [13]:
# check final DataFrame
pd.set_option("display.max_columns", None)
display(df_final.describe(include="all"))
pd.reset_option("display.max_columns")

,GEOID,town_name,population,valid_population,area_sq_km,pct_river_corridor,pct_below_poverty,percent_elderly,pct_no_vehicle,pct_renter_occupied,median_income,IFLD_EALT_weighted,EAL_per_capita,risk_eal,risk_eal_per_capita,risk_nri,need_eal,need_eal_per_capita,need_nri,gap_eal,gap_eal_per_capita,gap_nri,funding_total,funding_per_capita,claims_paid_per_capita,vulnerability,risk_eal_rel,risk_eal_per_capita_rel,risk_nri_rel,need_eal_rel,need_eal_per_capita_rel,need_nri_rel,gap_eal_rel,gap_eal_per_capita_rel,gap_nri_rel,funding_rel,claims_rel,vulnerability_rel,funding_rank,claims_rank,vulnerability_rank,risk_rank_eal,risk_rank_eal_per_capita,risk_rank_nri,need_rank_eal,need_rank_eal_per_capita,need_rank_nri,gap_rank_eal,gap_rank_eal_per_capita,gap_rank_nri,quadrant_eal,quadrant_eal_per_capita,quadrant_nri
count,2.560000e+02,257,257.000000,256,256.000000,257.000000,251.000000,251.000000,251.000000,251.000000,248.000000,2.560000e+02,256.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,2.510000e+02,2.510000e+02,251.000000,2.510000e+02,257.000000,257.000000,251.000000,2.500000e+02,2.500000e+02,2.500000e+02,2.500000e+02,2.500000e+02,2.500000e+02,2.500000e+02,2.500000e+02,250.000000,2.500000e+02,2.500000e+02,2.500000e+02,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,251.000000,250,250,250
unique,NaN,257,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,5,5
top,NaN,Addison town,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,zero_funding,zero_funding,zero_funding
freq,NaN,1,NaN,250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,130,130,130
mean,5.001514e+09,NaN,5035.844358,NaN,97.278278,3.518474,8.909979,23.944363,3.857002,16.897462,83175.865349,1.267566e+06,316.489267,0.503016,0.501349,0.501799,0.502591,0.501758,0.250899,-3.980879e-18,-7.961759e-18,-0.251000,6.174000e+05,155.791293,151.187889,0.502167,-5.329071e-18,1.776357e-18,-1.421085e-17,1.776357e-18,3.552714e-18,-1.421085e-17,-8.881784e-18,-2.309264e-17,-0.828580,1.776357e-17,-1.136868e-16,2.131628e-16,0.243984,0.293625,0.502120,0.501992,0.501992,0.501944,0.501880,0.501976,0.501944,0.501689,0.501944,0.501737,NaN,NaN,NaN
std,8.685967e+05,NaN,40402.771888,NaN,34.758921,2.506953,5.117351,7.630585,3.592786,11.114544,21084.022230,1.011149e+07,117.678380,0.289121,0.288847,0.288674,0.186076,0.176998,0.144337,2.717615e-01,2.916947e-01,0.290799,4.917189e+06,377.088973,390.428041,0.185903,5.761990e-01,5.761792e-01,5.761655e-01,3.709426e-01,3.532106e-01,5.761655e-01,1.251057e+00,1.205563e+00,0.961887,1.139853e+00,1.032414e+00,3.710306e-01,0.323156,0.332772,0.288676,0.288673,0.288663,0.288657,0.288678,0.288672,0.288657,0.288714,0.288674,0.288693,NaN,NaN,NaN
min,5.000100e+09,NaN,0.000000,NaN,3.813185,0.000000,0.000000,6.706793,0.000000,0.000000,13849.000000,2.625130e+04,110.977624,0.004000,0.004000,0.004000,0.034333,0.132000,0.002000,-6.860000e-01,-6.890000e-01,-0.956000,0.000000e+00,0.000000,0.000000,0.024667,-9.920319e-01,-9.920319e-01,-9.920319e-01,-9.316069e-01,-7.370518e-01,-9.920319e-01,-3.151685e+00,-2.841909e+00,-3.155865,-1.000000e+00,-1.000000e+00,-9.508632e-01,0.000000,0.000000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,NaN,NaN,NaN
25%,5.000731e+09,NaN,785.000000,NaN,82.562679,1.921692,5.289913,19.678549,1.125973,8.764779,70194.250000,2.401186e+05,228.450018,0.254000,0.253000,0.254000,0.359000,0.357000,0.127000,-1.923333e-01,-2.286667e-01,-0.473000,0.000000e+00,0.000000,0.000000,0.370667,-4.960159e-01,-4.990040e-01,-4.960159e-

In [14]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257 entries, 0 to 256
Data columns (total 53 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   GEOID                     256 non-null    float64
 1   town_name                 257 non-null    object 
 2   population                257 non-null    int64  
 3   valid_population          256 non-null    object 
 4   area_sq_km                256 non-null    float64
 5   pct_river_corridor        257 non-null    float64
 6   pct_below_poverty         251 non-null    float64
 7   percent_elderly           251 non-null    float64
 8   pct_no_vehicle            251 non-null    float64
 9   pct_renter_occupied       251 non-null    float64
 10  median_income             248 non-null    float64
 11  IFLD_EALT_weighted        256 non-null    float64
 12  EAL_per_capita            256 non-null    float64
 13  risk_eal                  251 non-null    float64
 14  risk_eal_p

In [15]:
df_final.columns

Index(['GEOID', 'town_name', 'population', 'valid_population', 'area_sq_km',
       'pct_river_corridor', 'pct_below_poverty', 'percent_elderly',
       'pct_no_vehicle', 'pct_renter_occupied', 'median_income',
       'IFLD_EALT_weighted', 'EAL_per_capita', 'risk_eal',
       'risk_eal_per_capita', 'risk_nri', 'need_eal', 'need_eal_per_capita',
       'need_nri', 'gap_eal', 'gap_eal_per_capita', 'gap_nri', 'funding_total',
       'funding_per_capita', 'claims_paid_per_capita', 'vulnerability',
       'risk_eal_rel', 'risk_eal_per_capita_rel', 'risk_nri_rel',
       'need_eal_rel', 'need_eal_per_capita_rel', 'need_nri_rel',
       'gap_eal_rel', 'gap_eal_per_capita_rel', 'gap_nri_rel', 'funding_rel',
       'claims_rel', 'vulnerability_rel', 'funding_rank', 'claims_rank',
       'vulnerability_rank', 'risk_rank_eal', 'risk_rank_eal_per_capita',
       'risk_rank_nri', 'need_rank_eal', 'need_rank_eal_per_capita',
       'need_rank_nri', 'gap_rank_eal', 'gap_rank_eal_per_capita',
       '

In [16]:
# export final DataFrame to CSV for web app
df_final.to_csv("../docs/static/resources/town_stats.csv", index=False)

In [17]:
# print IFLD_EALT_weighted sum for verification
print("Total IFLD_EALT_weighted:", df_merged["IFLD_EALT_weighted"].sum())
# print per capita IFLD_EALT_weighted
print(
    "Per capita IFLD_EALT_weighted:",
    df_merged["IFLD_EALT_weighted"].sum() / df_merged["population"].sum(),
)

Total IFLD_EALT_weighted: 162001159.98132646
Per capita IFLD_EALT_weighted: 250.3471764770014


In [18]:
df = pd.read_csv("../data/cleaned/town_level_merged_for_eda.csv")
print("IFLD_EALT_weighted sum:", df["IFLD_EALT_weighted"].sum())
print("non-null count:", df["IFLD_EALT_weighted"].notna().sum())
print("\nTop 10 towns by EAL:")
print(
    df[["town_name", "IFLD_EALT_weighted"]]
    .dropna()
    .sort_values("IFLD_EALT_weighted", ascending=False)
    .head(10)
    .to_string(index=False)
)
nri = pd.read_csv("../data/cleaned/fema_nri_town_level.csv")
print("\nNRI EAL cols:", [c for c in nri.columns if "EAL" in c.upper()])

IFLD_EALT_weighted sum: 162495846.433005
non-null count: 255

Top 10 towns by EAL:
            town_name  IFLD_EALT_weighted
      Burlington city        6.560160e+06
      Bennington town        4.275965e+06
     Brattleboro town        3.264232e+06
           Barre city        3.259148e+06
South Burlington city        3.194753e+06
      Montpelier city        3.040003e+06
         Rutland city        2.677038e+06
           Stowe town        2.291186e+06
       Williston town        2.231320e+06
          Milton town        2.215642e+06

NRI EAL cols: ['IFLD_EALT_weighted', 'IFLD_EALP_weighted', 'IFLD_EALPE_weighted', 'IFLD_EALB_weighted', 'IFLD_EALA_weighted', 'EAL_per_capita', 'EAL_per_building_value', 'EAL_per_agricultural_value']
